# Static Runtime Context 基本概念

静态运行时上下文（Static Runtime Context）表示一次 Agent 运行期间保持不变的依赖和元数据，例如：

- 当前用户 ID、租户 ID、用户角色和语言偏好
- 数据库连接、Repository、HTTP Client 等依赖对象
- 本次调用使用的权限、环境或功能开关

Context 通常在运行开始时通过 `invoke(..., context=...)` 或 `stream(..., context=...)` 传入。工具和 Middleware 可以读取它，但不应该在运行过程中修改它。

这里的“静态”强调的是**一次运行期间不变**。可以使用 `@dataclass(frozen=True)` 在 Python 层阻止字段被重新赋值，但这不代表其中引用的数据库连接等对象也会自动变成不可变对象。

## 与 Config、Checkpoint、Store 的区别

| 概念 | 主要职责 | 生命周期 | 常见内容 | 入口或访问方式 |
| --- | --- | --- | --- | --- |
| Static Runtime Context | 向本次运行注入只读业务身份与依赖 | 单次 `invoke/stream` | `user_id`、角色、数据库连接、Repository | `context=...`、`runtime.context` |
| Runnable Config | 控制本次 Runnable/Graph 如何执行 | 单次运行 | tags、metadata、recursion limit、callbacks、`thread_id` | `config=...`、工具中的 `runtime.config` |
| Checkpoint | 保存某个 thread 的 Agent State 快照 | 跨多次运行，同一 thread | 消息、Agent State、中断恢复位置 | checkpointer + `configurable.thread_id` |
| Store | 保存可跨 thread 访问的长期数据 | 取决于 Store 后端 | 用户画像、偏好、长期记忆、业务数据 | `runtime.store`、namespace + key |

可以用一句话区分：

- Context：**这次运行是谁、拥有什么依赖**。
- Config：**这次运行应该怎样执行**。
- Checkpoint：**这个会话执行到了哪里、当前状态是什么**。
- Store：**跨会话长期保存了什么**。

## 定义 Context Schema

`context_schema` 用于声明 Context 的结构。Context 不是 Agent State，不需要继承 `AgentState`；常用 dataclass、TypedDict 或其他类型来描述。

In [ ]:
from dataclasses import dataclass
from typing import Any


@dataclass(frozen=True)
class AppContext:
    user_id: str
    user_role: str
    locale: str = "zh-CN"
    db_connection: Any | None = None


run_context = AppContext(
    user_id="user-1",
    user_role="member",
    locale="zh-CN",
)

print(run_context)


## 在 Agent 中声明并传入 Context

将 Schema 传给 `create_agent(context_schema=...)`，调用时再通过独立的 `context` 参数传入实例。

Context **不会自动进入模型提示词**。仅仅传入 `user_id`，模型并不会自动知道它；必须在 Middleware 或工具中读取并使用。后面两个 Notebook 会分别展示这两种方式。

In [ ]:
import os

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model


load_dotenv(override=True)

model = init_chat_model(
    model="gpt-5.4-mini",
    model_provider="openai",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url=os.getenv("OPENROUTER_BASE_URL"),
)

agent = create_agent(
    model=model,
    tools=[],
    context_schema=AppContext,
)


In [ ]:
# 三类参数位于不同入口，职责彼此独立。
input_data = {"messages": "你好，请简单介绍你自己"}
run_config = {
    "tags": ["static-context-demo"],
    "configurable": {"thread_id": "thread-1"},
}

result = agent.invoke(
    input_data,
    config=run_config,
    context=run_context,
)

print(result["messages"][-1].content)


## 小结

- Context 是依赖注入通道，不是提示词、消息历史或持久化层。
- 相同 Agent 的不同调用可以传入不同 Context。
- Context 不会被 checkpoint 自动保存，也不应该用来保存不断变化的对话状态。
- 用户身份等可信信息应由应用传入 Context，而不是让模型从普通工具参数中自行选择。